# Multi-Label Classification - Enhanced Notebook
This notebook trains individual binary classifiers for each label in a multi-label classification setup.
We'll improve modularity, add visualization, and summarize results at the end.

In [1]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Reproducibility
def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)



In [2]:
import os
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import cv2
import numpy as np
from glob import glob
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers
from tensorflow import keras 
import tensorflow as tf

## Load and Explore Dataset

In [3]:
import glob
import pandas as pd
from PIL import Image
import numpy as np

def process_images(path, label, group_name, df, counter, images, targets, load_img):
    for ext in ["*.png", "*.webp", "*.jpg"]:
        for filename in glob.glob(path + ext, recursive=True):
            images.append(filename)
            df.loc[counter, 'img_path'] = filename
            df.loc[counter, group_name] = label
            counter += 1
            img = Image.open(filename)
            img = img.resize((256, 256))
            if img is not None:
                rgb = img.convert('RGB')
                if rgb is not None:
                    targets.append(1 if label == 'true' else 0)
                    load_img.append(np.array(rgb))
    return counter

def load(path_true, path_false, group_name, df, counter):
    images, targets, load_img = [], [], []
    
    # Process true images
    counter = process_images(path_true, 'true', group_name, df, counter, images, targets, load_img)
    number_images_true = len(images)
    
    # Process false images
    counter = process_images(path_false, 'false', group_name, df, counter, images, targets, load_img)
    number_images_false = len(images) - number_images_true
    
    return number_images_true, number_images_false, images, targets, load_img, counter

# Example usage:
# df = pd.DataFrame()
# counter = 0
# number_images_true, number_images_false, images, targets, load_img, counter = load("path/to/true/", "path/to/false/", "group_name", df, counter)



In [4]:

import pandas as pd

def load_combined_dataframe(file_paths):
    """
    Load and merge specific label CSVs into one DataFrame using file_paths dict.
    Drops rows with any missing values in any column.
    """

    keys_to_load = list(file_paths.keys())  # ← load all keys dynamically, or keep your fixed list

    dfs = []
    for key in keys_to_load:
        path = file_paths.get(key)
        if path:
            try:
                df = pd.read_csv(path)

                # Drop rows that contain any missing values
                df_clean = df.dropna()

                if not df_clean.empty:
                    dfs.append(df_clean)
                else:
                    print(f"⚠️ {key} loaded but is empty after dropna()")
            except Exception as e:
                print(f"❌ Failed to load {key}: {e}")
        else:
            print(f"⚠️ No path found for key: {key}")

    if not dfs:
        raise ValueError("No CSVs were loaded successfully or all were empty after dropna().")

    combined_df = pd.concat(dfs, ignore_index=True)

    # Ensure all expected columns are present
    expected_columns = [
        'img_path', 'bullets', 'interesting', 'readable',
        'agenda', 'tables', 'positioning', 'pictures',
        'infographics', 'graphics', 'size_of_text'
    ]

    for col in expected_columns:
        if col not in combined_df.columns:
            combined_df[col] = 0  # Fill missing expected columns with default 0

    return combined_df[expected_columns]


data_dir = "../data"
file_paths = {
    "graphics_real": os.path.join(data_dir, "graphicsReal.csv"),
    "graphics_not_real": os.path.join(data_dir, "graphicsNotReal.csv"),
    "bullets_real": os.path.join(data_dir, "bulletsReal.csv"),
    "bullets_not": os.path.join(data_dir, "NoBullets.csv"),
    "readable": os.path.join(data_dir, "readable.csv"),
    "not_readable": os.path.join(data_dir, "NoReadable.csv"),
    "boring": os.path.join(data_dir, "boring.csv"),
    "interesting": os.path.join(data_dir, "interesting.csv"),
    "agenda_yes": os.path.join(data_dir, "agendaYes.csv"),
    "text_size_ok": os.path.join(data_dir, "sizeOfTextOk.csv"),
    "text_size_not_ok": os.path.join(data_dir, "sizeOfTextNotOk.csv"),
    "positioning_ok": os.path.join(data_dir, "positioningOk.csv"),
    "positioning_not_ok": os.path.join(data_dir, "positioningNotOk.csv"),
    "pictures": os.path.join(data_dir, "pictures.csv"),
    "no_pictures": os.path.join(data_dir, "noPictures.csv"),
    "tables": os.path.join(data_dir, "tables.csv"),
    "infographics": os.path.join(data_dir, "infographics.csv"),
    "no_infographics": os.path.join(data_dir, "noInfographics.csv"),
    "mixed": os.path.join(data_dir, "mixed.csv")
}
df = load_combined_dataframe(file_paths)


In [5]:
df

,img_path,bullets,interesting,readable,agenda,tables,positioning,pictures,infographics,graphics,size_of_text
0,../data/ppt/graphics/has/10.PNG,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,../data/ppt/graphics/has/16.PNG,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,../data/ppt/graphics/has/17.PNG,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,../data/ppt/graphics/has/1g.PNG,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0
4,../data/ppt/graphics/has/2g.PNG,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
1828,../data/ppt/other/2DDXORTPL7KXJI7E5HBRGSTTOTXV...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1829,../data/ppt/other/2DDXORTPL7KXJI7E5HBRGSTTOTXV...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
1830,../data/ppt/other/2DDXORTPL7KXJI7E5HBRGSTTOTXV...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
1831,../data/ppt/other/2DDXORTPL7KXJI7E5HBRGSTTOTXV...,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0


## Data Preprocessing

In [6]:
#!pip install ImageHash
import glob
from collections import defaultdict
from PIL import Image
import imagehash

def detect_duplicates_in_directories(paths):
    all_hashes = defaultdict(list)
    duplicates = []

    for path in paths:
        for filename in glob.glob(path + "*.png", recursive=True):
            img = Image.open(filename)
            img_hash = str(imagehash.phash(img))  # Compute perceptual hash
            all_hashes[img_hash].append(filename)

    # Find duplicates
    for img_hash, filenames in all_hashes.items():
        if len(filenames) > 1:
            duplicates.append(filenames)


    return duplicates

# Example usage
directories_to_check = [
    "../data/ppt/bullets/has_bullets/",
    "../data/ppt/bullets/no_bullets/",
    "../data/ppt/graphics/has/",
    "../data/ppt/graphics/hasnot/",
    "../data/ppt/readable/yes/",
    "../data/ppt/readable/no/",
    "../data/ppt/infographics/with/",
    "../data/ppt/infographics/without/",
    "../data/ppt/pictures/with/",
    "../data/ppt/pictures/without/",
    "../data/ppt/positioning/good/",
    "../data/ppt/positioning/bad/",
    "../data/ppt/boring_interesting/interesting_slides/",
    "../data/ppt/boring_interesting/boring_slides/",
    "../data/ppt/size_of_text/ok/",
    "../data/ppt/size_of_text/not_ok/"
]

duplicate_pairs = detect_duplicates_in_directories(directories_to_check)
print("Near duplicate image pairs:", duplicate_pairs)


Near duplicate image pairs: []


In [7]:
import os
import glob
from collections import defaultdict
from PIL import Image
import imagehash

def remove_duplicate_files_in_directories(paths):
    all_hashes = defaultdict(list)

    # Compute perceptual hash for each image in all directories
    for path in paths:
        for filename in glob.glob(path + "*.png", recursive=True):
            img = Image.open(filename)
            img_hash = str(imagehash.phash(img))
            all_hashes[img_hash].append(filename)

    # Remove duplicate files
    removed_files = []
    for img_hash, filenames in all_hashes.items():
        if len(filenames) > 1:
            # Keep the first occurrence, delete the rest
            for duplicate_file in filenames[1:]:
                os.remove(duplicate_file)
                removed_files.append(duplicate_file)

    return removed_files

# Example usage
directories_to_check = [
    "../data/ppt/bullets/has_bullets/",
    "../data/ppt/bullets/no_bullets/",
    "../data/ppt/graphics/has/",
    "../data/ppt/graphics/hasnot/",
    "../data/ppt/readable/yes/",
    "../data/ppt/readable/no/",
    "../data/ppt/infographics/with/",
    "../data/ppt/infographics/without/",
    "../data/ppt/pictures/with/",
    "../data/ppt/pictures/without/",
    "../data/ppt/positioning/good/",
    "../data/ppt/positioning/bad/",
    "../data/ppt/boring_interesting/interesting_slides/",
    "../data/ppt/boring_interesting/boring_slides/",
    "../data/ppt/size_of_text/ok/",
    "../data/ppt/size_of_text/not_ok/"
]

removed_files = remove_duplicate_files_in_directories(directories_to_check)

def save_removed_files_to_txt(removed_files, output_file):
    with open(output_file, 'w') as f:
        for file_path in removed_files:
            f.write(file_path + '\n')

import pandas as pd

def filter_removed_files_and_save(file_paths, removed_files):
    """
    Filters out rows from each CSV in file_paths where 'img_path' is in removed_files.
    Saves the cleaned DataFrames back to their original paths.
    
    Parameters:
    - file_paths: dict of {name: path_to_csv}
    - removed_files: list of file paths to exclude
    """
    for name, path in file_paths.items():
        try:
            df = pd.read_csv(path)
            original_shape = df.shape

            if 'img_path' not in df.columns:
                print(f"⚠️ Skipping {name}: 'img_path' column missing.")
                continue

            filtered_df = df[~df['img_path'].isin(removed_files)]
            print(f"{name}: {original_shape} → {filtered_df.shape} (removed {original_shape[0] - filtered_df.shape[0]} rows)")
            
            # Save back to the same file
            filtered_df.to_csv(path, index=False)

        except Exception as e:
            print(f"❌ Failed to process {name}: {e}")



In [8]:
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.utils import img_to_array

def load_img_from_file(path):
    img = Image.open(path)
    img = img.resize((256, 256))
    img = img.convert('RGB')
    return np.array(img)
    # if img is not None:
    #     gray = img.convert('L')
    #     return np.array(gray)
    # return None


# Load ResNet50 model once
embedding_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')


def calculate_embeddings(image_paths, target_size=(256, 256)):
    """
    Calculate embeddings for a list of image paths using ResNet50.

    Parameters:
        image_paths (list): List of image file paths.
        target_size (tuple): Size to resize each image to (default: (224, 224)).

    Returns:
        numpy.ndarray: Array of shape (N, 2048) with image embeddings.
    """
    embeddings = []

    for path in image_paths:
        try:
            if not os.path.exists(path):
                print(f"❌ File not found: {path}")
                continue

            # Load image and convert to array
            img = load_img_from_file(path)
            img_array = img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0)
            img_array = preprocess_input(img_array)

            # Predict embedding
            embedding = embedding_model.predict(img_array, verbose=0)
            embeddings.append(embedding.squeeze())

        except Exception as e:
            print(f"⚠️ Failed to process {path}: {e}")

    if not embeddings:
        print("🚫 No valid embeddings generated.")
        return np.empty((0, 2048))  # Return empty array with correct shape

    return np.array(embeddings)


def calculate_visual_differences(embeddings):
    """
    Calculate pairwise cosine similarity between image embeddings to quantify visual differences.
    
    Parameters:
        embeddings (numpy.ndarray): Array of image embeddings.
        
    Returns:
        numpy.ndarray: Pairwise cosine similarity matrix.
    """
    # Reshape embeddings for cosine similarity calculation
    embeddings = np.squeeze(embeddings)

    # Calculate pairwise cosine similarity
    similarities = cosine_similarity(embeddings)

    return similarities

def extract_visually_different_images(merged_df, num_images):
    """
    Extract visually different images from the merged DataFrame.
    
    Parameters:
        merged_df (DataFrame): The merged DataFrame containing the data.
        num_images (int): The number of visually different images to extract.
        
    Returns:
        DataFrame: DataFrame containing the extracted visually different images.
    """
    # Calculate embeddings for images in merged_df
    embeddings = calculate_embeddings(merged_df['img_path'].tolist())

    # Calculate pairwise cosine similarity between embeddings
    similarities = calculate_visual_differences(embeddings)

    # Compute average cosine similarity for each image
    avg_similarities = np.mean(similarities, axis=1)

    # Get indices of top num_images images with highest average similarity
    top_indices = np.argsort(avg_similarities)[-num_images:]

    # Extract visually different images from merged_df
    visually_different_df = merged_df.iloc[top_indices]

    return visually_different_df

def balance_dataframe(merged_df, column):
    """
    Balance the occurrences of each category in a specified column of the DataFrame
    by selecting visually different samples with the same number of occurrences for each category.

    Parameters:
        merged_df (DataFrame): The input DataFrame containing the data.
        column (str): The column name representing the categories.

    Returns:
        DataFrame: The balanced DataFrame with visually different occurrences for each category.
    """
    # Count the occurrences of each value in the column
    value_counts = merged_df[column].value_counts()
    min_count = min(value_counts)
    
    # Extract balanced samples
    balanced_df_0 = extract_visually_different_images(merged_df[merged_df[column] == 0], min_count)
    balanced_df_1 = extract_visually_different_images(merged_df[merged_df[column] == 1], min_count)

    # Concatenate and shuffle
    balanced_df = pd.concat([balanced_df_0, balanced_df_1], ignore_index=True)
    balanced_df = balanced_df.sample(frac=1).reset_index(drop=True)

    # Print statistics
    num_positives = (balanced_df[column] == 1).sum()
    num_negatives = (balanced_df[column] == 0).sum()
    total = len(balanced_df)

    print(f"✅ Balanced dataset created:")
    print(f" - Positives (1): {num_positives}")
    print(f" - Negatives (0): {num_negatives}")
    print(f" - Total samples: {total}")
    print("value_counts:", value_counts)
    print("min_count:", min_count)

    return balanced_df,num_positives,num_negatives,total




In [10]:
label_columns = ['bullets','interesting','readable','agenda','size_of_text','tables','positioning', 'pictures','infographics','graphics']
group_configs = [
            ("bullets", "D:/WorkspaceAI/ppt/bullets/has_bullets/", "D:/WorkspaceAI/ppt/bullets/no_bullets/"),
            ("graphics", "D:/WorkspaceAI/ppt/graphics/has/", "D:/WorkspaceAI/ppt/graphics/hasnot/"),
            ("infographics", "D:/WorkspaceAI/ppt/infographics/with/", "D:/WorkspaceAI/ppt/infographics/without/"),
            ("pictures", "D:/WorkspaceAI/ppt/pictures/with/", "D:/WorkspaceAI/ppt/pictures/without/"),
            ("positioning", "D:/WorkspaceAI/ppt/positioning/good/", "D:/WorkspaceAI/ppt/positioning/bad/"),
            ("readable", "D:/WorkspaceAI/ppt/readable/yes/", "D:/WorkspaceAI/ppt/readable/no/"),
            ("interesting", "D:/WorkspaceAI/ppt/boring_interesting/interesting_slides/", "D:/WorkspaceAI/ppt/boring_interesting/boring_slides/"),
            ("size_of_text", "D:/WorkspaceAI/ppt/size_of_text/ok/", "D:/WorkspaceAI/ppt/size_of_text/not_ok/"),
            ("tables", "D:/WorkspaceAI/ppt/tables/have/", "D:/WorkspaceAI/ppt/tables/dont_have/"),
            ("agenda", "D:/WorkspaceAI/ppt/agenda/yes/", "D:/WorkspaceAI/ppt/agenda/no/")
        ]

from enum import Enum

class Model(Enum):
    LOGISTIC_REGRESSION = "Logistic Regression"
    DECISION_TREE = "Decision Tree"
    RANDOM_FOREST = "Random Forest"
    SUPPORT_VECTOR_MACHINE = "Support Vector Machine"
    K_NEAREST_NEIGHBORS = "K-Nearest Neighbors"
    NAIVE_BAYES = "Naive Bayes"
    NEURON_NETWORK = "Neuron Network"
    CONV_NEURON_NETWORK = "Convolutional Neuron Network"
    # Add more models here as needed

def load_single_image(img_path):
    img = Image.open(img_path)
    img = img.resize((256, 256))
    gray = img.convert('RGB')
    if gray is not None:
        return np.array(gray)
    return None
    
def load_images_from_df(df):
    images = []
    for path in df['img_path']:
        images.append(load_single_image(path))
    return images
    
def preprocess_data(df, label_columns, group_configs):
    model_data_map = {}
    #delete duplicates

    file_paths = {group_name: os.path.join("../data/", f"{group_name}.csv") for group_name, _, _ in group_configs}
    data_dir = "../data/logs"
    if not removed_files:
        print("✅ No files were removed.")
    else:
        print(f"❌ {len(removed_files)} files were removed:")
        for f in removed_files:
            print(f"- {f}")
        filter_removed_files_and_save(file_paths, removed_files)
        output_file_path = data_dir + "removed_files.txt"
        save_removed_files_to_txt(removed_files, output_file_path)
        print("List of removed files saved to:", output_file_path)
    #balanced dataset
    #label_columns = ['bullets','interesting','readable','agenda','size_of_text','tables','positioning', 'pictures','infographics','graphics']
    for x in label_columns:
        df_temp = df[df['img_path'].str.len() >= 15]
        print(f"🔍 Processing {x}, number of samples: {len(df_temp)}")
        df_balanced,num_positives,num_negatives,total = balance_dataframe(df_temp, x)
        images_df = load_images_from_df(df_balanced)
        model_data_map[x] = {
            'images_df': images_df,
            'df_balanced': df_balanced,
            'target': df_balanced[x],
            'img_path': df_balanced['img_path'],
            'model_type': Model.CONV_NEURON_NETWORK,
            'pos_example': num_positives,
            'num_negatives': num_negatives,
            'total': total,
            'evaluation': None
        }
    return model_data_map
    
model_data_map = preprocess_data(df, label_columns, group_configs)


✅ No files were removed.
🔍 Processing bullets, number of samples: 1833
✅ Balanced dataset created:
 - Positives (1): 768
 - Negatives (0): 768
 - Total samples: 1536
value_counts: bullets
0.0    1065
1.0     768
Name: count, dtype: int64
min_count: 768
🔍 Processing interesting, number of samples: 1833
✅ Balanced dataset created:
 - Positives (1): 885
 - Negatives (0): 885
 - Total samples: 1770
value_counts: interesting
0.0    948
1.0    885
Name: count, dtype: int64
min_count: 885
🔍 Processing readable, number of samples: 1833
✅ Balanced dataset created:
 - Positives (1): 763
 - Negatives (0): 763
 - Total samples: 1526
value_counts: readable
1.0    1070
0.0     763
Name: count, dtype: int64
min_count: 763
🔍 Processing agenda, number of samples: 1833
✅ Balanced dataset created:
 - Positives (1): 615
 - Negatives (0): 615
 - Total samples: 1230
value_counts: agenda
0.0    1218
1.0     615
Name: count, dtype: int64
min_count: 615
🔍 Processing size_of_text, number of samples: 1833
✅ Bala

## Model Architecture

In [ ]:
########### NN model

In [18]:
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping #Early stopping is a technique used to prevent overfitting by monitoring the validation loss during training and stopping the training process when the validation loss starts to increase

# Define the early stopping callback
early_stopping = EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)

def create_nn_model(learning_rate=0.001, dropout_rate=0.0, num_dense_layers=1, num_units=64):
    model = Sequential()
    model.add(Flatten(input_shape=(256, 256, 3)))
    
    for i in range(num_dense_layers-1):
        model.add(Dense(num_units / (i+1), activation='relu'))
        model.add(Dropout(dropout_rate))  # Regularization
    
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

def create_nn_model_2(learning_rate=0.001, dropout_rate=[[0.0]], num_dense_layers=1, num_units=64):
    model = Sequential()
    model.add(Flatten(input_shape=(256, 256, 3)))
    
    for i in range(num_dense_layers-1):
        model.add(Dense(num_units / (i+1), activation='relu'))
        model.add(Dropout(dropout_rate[i][i]))  # Regularization
    
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

def nn_model_with_tuning(X_train_reshaped, y_train, X_val_reshaped, y_val):
    # Normalize features
    
    # Create KerasClassifier model
    keras_model = KerasClassifier(build_fn=create_nn_model, verbose=0, callbacks=[early_stopping])
    
    # Define hyperparameters grid
    param_grid = {
        'epochs': [ 100, 150, 200],
        'batch_size': [16, 32, 64],
        'learning_rate': [0.001, 0.01, 0.1],
        'dropout_rate': [0.0, 0.05, 0.1, 0.2],
        'num_dense_layers': [ 3, 4],  # Number of dense layers
        'num_units': [32, 64, 128 ]  # Number of units in first/upper dense layer
    }
    #     'epochs': [50, 100, 150, 200],
    #     'batch_size': [16, 32, 64],
    #     'learning_rate': [0.001, 0.01, 0.1],
    #     'dropout_rate': [0.0, 0.05, 0.1, 0.2],
    #     'num_dense_layers': [1, 2, 3, 4, 5],  # Number of dense layers
    #     'num_units': [32, 64, 128, 220 ]  # Number of units in first/upper dense layer
    # }

    # Perform Grid Search
    grid_search = GridSearchCV(estimator=keras_model, param_grid=param_grid, scoring='accuracy', cv=5, verbose=0)
    grid_search.fit(X_train_reshaped, y_train, validation_data=(X_val_reshaped, y_val))
    
    # Get the best model from Grid Search
    best_model = grid_search.best_estimator_
    print("Best: %f using %s" % (grid_search.best_score_, grid_search.best_params_))
    return best_model

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam

def nn_model(X_train, y_train, X_val, y_val, X_test, y_test):

    # Normalize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # Reshape data for Convolutional Neural Network (CNN)
    X_train_reshaped = X_train_scaled.reshape(-1, 256, 256, 3)
    X_val_reshaped = X_val_scaled.reshape(-1, 256, 256, 3)
    X_test_reshaped = X_test_scaled.reshape(-1, 256, 256, 3)
    
    best_model = nn_model_with_tuning(X_train_reshaped, y_train, X_val_reshaped, y_val)

    # Evaluate model
    y_pred = (best_model.predict(X_test_reshaped) > 0.5).astype("int32")

    return best_model, y_pred


In [19]:
########### CNN model

In [20]:
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, hamming_loss,
    jaccard_score, confusion_matrix, cohen_kappa_score
)

def evaluate_model_performance(y_test, y_pred, group=None):
    evaluationMap = {}
    
    # Accuracy
    accuracy = accuracy_score(y_test, y_pred)
    evaluationMap['accuracy'] = accuracy

    # Hamming Loss
    hamming = hamming_loss(y_test, y_pred)
    evaluationMap['hamming'] = hamming

    # Jaccard Similarity
    try:
        jaccard = jaccard_score(y_test, y_pred, average='binary')
    except:
        jaccard = None
    evaluationMap['jaccard'] = jaccard

    # F1 Score
    f1 = f1_score(y_test, y_pred)
    evaluationMap['f1'] = f1

    # ROC AUC Score
    try:
        auc_scores = roc_auc_score(y_test, y_pred)
    except:
        auc_scores = None
    evaluationMap['auc_scores'] = auc_scores

    # Cohen's Kappa
    kappa_scores = cohen_kappa_score(y_test, y_pred)
    evaluationMap['kappa_scores'] = kappa_scores

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    evaluationMap['confusion_matrix'] = cm

    # Optional logging
    if group:
        print(f"\nEvaluation for group: {group}")
        print("Confusion Matrix:\n", cm)
        print("Accuracy:", accuracy)
        print("F1 Score:", f1)
        print("Hamming Loss:", hamming)
        print("Jaccard Similarity:", jaccard)
        print("ROC AUC Score:", auc_scores)
        print("Cohen's Kappa:", kappa_scores)

    return evaluationMap


In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping #Early stopping is a technique used to prevent overfitting by monitoring the validation loss during training and stopping the training process when the validation loss starts to increase
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Activation


# Define the early stopping callback
early_stopping = EarlyStopping(monitor='accuracy', patience=3, restore_best_weights=True)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

def create_fine_tuned_cnn(input_shape=(256, 256, 3), learning_rate=0.001, dropout_rate=0.5):
    model = Sequential()

    # Conv Block 1
    model.add(Conv2D(32, (3, 3), padding='same', input_shape=input_shape))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # Conv Block 2
    model.add(Conv2D(64, (3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # Conv Block 3
    model.add(Conv2D(64, (3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # Conv Block 4
    model.add(Conv2D(64, (3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # Flatten
    model.add(Flatten())

    # Fully Connected 1
    model.add(Dense(512))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Dropout(dropout_rate))

    # Fully Connected 2
    model.add(Dense(256))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Dropout(dropout_rate))

    # Output Layer
    model.add(Dense(1, activation='sigmoid'))

    # Compile the model
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

    return model

def create_cnn_model(input_shape=(256, 256, 3), num_classes=1, learning_rate=0.001, dropout_rate=0.5, num_conv_layers=4, initial_filters=32, initial_dense_units=512):
    model = Sequential()
    
    # Add initial Conv2D layer with input shape
    model.add(Conv2D(initial_filters, (3, 3), padding='same', input_shape=input_shape))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D((2, 2)))
    
    # Add additional Conv2D and MaxPooling2D layers in a loop
    filters = initial_filters
    for i in range(1, num_conv_layers):
        filters *= 2
        model.add(Conv2D(filters, (3, 3), padding='same'))
        model.add(BatchNormalization())
        model.add(Activation('relu'))
        model.add(MaxPooling2D((2, 2)))
    
    model.add(Flatten())
    
    # Add fully connected layers
    model.add(Dense(initial_dense_units))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Dropout(dropout_rate))  # Regularization
    
    model.add(Dense(initial_dense_units // 2))  # Ensure integer division
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Dropout(dropout_rate))  # Regularization
    
    # Output Layer with num_classes units
    model.add(Dense(num_classes, activation='sigmoid'))
    
    # Compile the model
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
import numpy as np

def run_cross_validation(X, y, model, n_splits=3, epochs=50, batch_size=32, patience=5, verbose_param=0):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    histories = []
    scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n📂 Fold {fold + 1}/{n_splits}")

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        early_stopping = EarlyStopping(
            monitor='val_loss',
            patience=patience,
            restore_best_weights=True,
            verbose=verbose_param
        )

        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            verbose=0,  # ✅ no console output during training
            callbacks=[early_stopping]
        )

        val_preds = (model.predict(X_val, verbose=0) > 0.5).astype("int32")
        acc = accuracy_score(y_val, val_preds)
        print(f"✅ Fold {fold + 1} Accuracy: {acc:.4f}")

        histories.append(history)
        scores.append(acc)

        # Free memory
        K.clear_session()

    print(f"\n🎯 Average Accuracy over {n_splits} folds: {np.mean(scores):.4f} ± {np.std(scores):.4f}")
    return histories, scores, model


def cnn_model_without_tuning(X_train, y_train, X_val, y_val):
    import numpy as np
    
    X = np.concatenate([X_train, X_val], axis=0)
    y = np.concatenate([y_train, y_val], axis=0)
    model = create_fine_tuned_cnn(input_shape=(256, 256, 3), learning_rate=0.001, dropout_rate=0.5)
    print("cross validation")
    histories, scores, model = run_cross_validation(X, y, model, n_splits=3, epochs=200, batch_size=16, patience=5, verbose_param=0)
    return model
    
def cnn_model_with_tuning(X_train, y_train, X_val, y_val):
    # Early stopping callback
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    # Wrap the model using KerasClassifier
    model = KerasClassifier(build_fn=create_cnn_model, input_shape=(256, 256, 3), num_classes=1, verbose=0)
    
    # Define hyperparameters grid
    param_grid = {
        'epochs': [100, 200],
        'batch_size': [8], 
        'learning_rate': [0.01],
        'dropout_rate': [0.3, 0.5],
        'num_conv_layers': [3, 4],
        'initial_filters': [32, 64],
        'initial_dense_units':[ 256, 512]                 
    }

#     Best: 0.844740 using {'batch_size': 32, 'dropout_rate': 0.0, 'epochs': 200, 'learning_rate': 0.001, 'num_conv_layers': 2, 'num_dense_units': 128, 'num_filters': 64}
# CONV_NEURON_NETWORK Accuracy of group positioning : 0.9032258064516129

    # Perform Grid Search
    grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring='accuracy', n_jobs=-1,verbose=0, cv=3)
    
    # Fit the model with grid search
    grid_result = grid.fit(X_train, y_train, validation_data=(X_val, y_val), callbacks=[early_stopping])

    
    # Get the best model from Grid Search
    best_model = grid_search.best_estimator_
    print("Best: %f using %s" % (grid_search.best_score_, grid_search.best_params_))
    return best_model

import numpy as np
from sklearn.metrics import accuracy_score

def find_best_threshold(y_probs,y_val):
    median = np.median(y_probs)
    print(median)

    thresholds = [
        0.4,
        0.45,
        0.5
    ]

    best_acc = 0
    best_threshold = 0.5

    for t in thresholds:
        y_pred = (y_probs > t).astype("int32")
        acc = accuracy_score(y_val, y_pred)
        print(f"Threshold: {t:.3f} -> Accuracy: {acc:.4f}")
        if acc > best_acc:
            best_acc = acc
            best_threshold = t

    print(f"\n✅ Best threshold: {best_threshold:.3f} with Accuracy: {best_acc:.4f}")
    return best_threshold, best_acc


import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam

def cnn_model(X_train, y_train, X_val, y_val, X_test, y_test, group):

    # Normalize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.fit_transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # Reshape data for Convolutional Neural Network (CNN)
    X_train_reshaped = X_train_scaled.reshape(-1, 256, 256, 3)
    X_val_reshaped = X_val_scaled.reshape(-1, 256, 256, 3)
    X_test_reshaped = X_test_scaled.reshape(-1, 256, 256, 3)

    print("build a model and train it")
    best_model = cnn_model_without_tuning(X_train_reshaped, y_train, X_val_reshaped, y_val)
    print("predict with model")

    # Find best threshold 
    y_probs = best_model.predict(X_test_reshaped)
    best_threshold, _ = find_best_threshold(y_probs, y_test)
    
    # Evaluate model
    y_pred = (y_probs > best_threshold).astype("int32")
    evaluationMap = evaluate_model_performance(y_test, y_pred, group = group)
    
    return best_model, y_pred, evaluationMap

In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import hamming_loss, jaccard_score
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report
from sklearn.metrics import cohen_kappa_score
from tensorflow.keras.utils import plot_model

def get_train_model(images, targets, images_paths, model_enum, group):
   # x_train, y_train, x_test, y_test, x_test_img_paths = shuffle_and_split_data(images, targets, images_paths, test_ratio=0.25)
    x_train, y_train, x_val, y_val, x_test, y_test, x_train_img_paths, x_val_img_paths, x_test_img_paths = shuffle_and_split_data(
    images, targets, images_paths, test_ratio=0.2, val_ratio=0.1)
    print("Data train shape:", x_train.shape, y_train.shape)
    print("Data val shape:", x_val.shape, y_val.shape)
    print("Data test shape:", x_test.shape, y_test.shape)
    # Flatten the 2D array to 1D for all models
    x_train_flat = x_train.reshape(x_train.shape[0], -1)
    x_val_flat = x_val.reshape(x_val.shape[0], -1)
    x_test_flat = x_test.reshape(x_test.shape[0], -1)
    y_pred = []
    evaluationMap={}
    if model_enum == Model.NEURON_NETWORK:
        model, y_pred = nn_model( x_train_flat, y_train, x_val_flat, y_val, x_test_flat, y_test)
    elif model_enum == Model.CONV_NEURON_NETWORK:
        model, y_pred,evaluationMap = cnn_model( x_train_flat, y_train, x_val_flat, y_val, x_test_flat, y_test,group)
    else:
        # Create model based on model_enum
        if model_enum == Model.LOGISTIC_REGRESSION:
            model = LogisticRegression()
        elif model_enum == Model.DECISION_TREE:
            model = DecisionTreeClassifier()
        elif model_enum == Model.RANDOM_FOREST:
            model = RandomForestClassifier()
        elif model_enum == Model.SUPPORT_VECTOR_MACHINE:
            model = SVC()
        elif model_enum == Model.K_NEAREST_NEIGHBORS:
            model = KNeighborsClassifier()
        elif model_enum == Model.NAIVE_BAYES:
            model = GaussianNB()

        # Train model and predict labels
        model.fit(x_train_flat, y_train)
        
        y_pred = model.predict(x_test_flat)

    return model, y_pred, y_test, evaluationMap


In [23]:
import numpy as np
from sklearn.model_selection import train_test_split

def shuffle_and_split_data(images, targets, image_paths, test_ratio=0.2, val_ratio=0.2):
    """
    Shuffle and split data into training, validation, and test sets with stratified class distribution.

    Parameters:
    - images: List or array of images.
    - targets: 1D array-like of class labels (binary or multiclass).
    - image_paths: List or array of image paths.
    - test_ratio: Proportion of the total data to include in the test split.
    - val_ratio: Proportion of the total data to include in the validation split.

    Returns:
    - x_train, y_train, x_val, y_val, x_test, y_test
    - paths_train, paths_val, paths_test
    """
    if not 0 <= test_ratio <= 1:
        raise ValueError("test_ratio must be between 0 and 1.")
    if not 0 <= val_ratio <= 1:
        raise ValueError("val_ratio must be between 0 and 1.")
    if test_ratio + val_ratio >= 1:
        raise ValueError("test_ratio + val_ratio must be less than 1.")

    # Convert to numpy arrays
    x_data = np.array(images)
    y_data = np.array(targets)
    paths = np.array(image_paths)

    # First split: train_val vs test
    x_train_val, x_test, y_train_val, y_test, paths_train_val, paths_test = train_test_split(
        x_data, y_data, paths, test_size=test_ratio, stratify=y_data, random_state=42
    )

    # Adjust validation ratio relative to the remaining (train_val) set
    val_ratio_adjusted = val_ratio / (1 - test_ratio)

    # Second split: train vs val
    x_train, x_val, y_train, y_val, paths_train, paths_val = train_test_split(
        x_train_val, y_train_val, paths_train_val,
        test_size=val_ratio_adjusted, stratify=y_train_val, random_state=42
    )

    return (
        x_train, y_train,
        x_val, y_val,
        x_test, y_test,
        paths_train, paths_val, paths_test
    )


In [24]:
def build_and_train_model(model_data_map):
    import gc
    import time
#for x in ['bullets','interesting','readable','agenda','size_of_text','tables','positioning', 'pictures','infographics','graphics']:
    for x in ['bullets','interesting','readable','agenda','size_of_text','tables','positioning', 'pictures','infographics','graphics']:
        # Force garbage collection
        gc.collect()
        time.sleep(10)
        print("################ MODEL "+str(x)+" #################")
        merged_df = model_data_map[x]['images_df']
        df_balanced = model_data_map[x]['df_balanced']
        images_df = model_data_map[x]['images_df']
        model_data_map[x]["model"],model_data_map[x]["y_test_pred"], model_data_map[x]["y_test"],model_data_map[x]["evaluation"]  = get_train_model(images_df, df_balanced[x], df_balanced['img_path'], model_data_map[x]["model_type"], x)

build_and_train_model(model_data_map)

################ MODEL bullets #################
Data train shape: (1074, 256, 256, 3) (1074,)
Data val shape: (154, 256, 256, 3) (154,)
Data test shape: (308, 256, 256, 3) (308,)
build a model and train it
cross validation

📂 Fold 1/3
✅ Fold 1 Accuracy: 0.8220

📂 Fold 2/3
✅ Fold 2 Accuracy: 0.9144

📂 Fold 3/3
✅ Fold 3 Accuracy: 0.9144

🎯 Average Accuracy over 3 folds: 0.8836 ± 0.0436
predict with model
10/10 [==============================] - 0s 38ms/step
0.43207318
Threshold: 0.400 -> Accuracy: 0.8506
Threshold: 0.450 -> Accuracy: 0.8474
Threshold: 0.500 -> Accuracy: 0.8474

✅ Best threshold: 0.400 with Accuracy: 0.8506

Evaluation for group: bullets
Confusion Matrix:
 [[130  24]
 [ 22 132]]
Accuracy: 0.8506493506493507
F1 Score: 0.8516129032258064
Hamming Loss: 0.14935064935064934
Jaccard Similarity: 0.7415730337078652
ROC AUC Score: 0.8506493506493507
Cohen's Kappa: 0.7012987012987013
################ MODEL interesting #################
Data train shape: (1239, 256, 256, 3) (1239,)

In [ ]:
## Save the models

In [25]:
import os

def save_all_models(model_data_map, save_dir="saved_models"):
    os.makedirs(save_dir, exist_ok=True)
    
    for category, model_data in model_data_map.items():
        model = model_data["model"]
        model_path = os.path.join(save_dir, f"{category}_model")
        model.save(model_path)
        print(f"Saved model for category '{category}' at '{model_path}'")
        
save_dir="../exported_model"
save_all_models(model_data_map,save_dir)


INFO:tensorflow:Assets written to: ../exported_model\bullets_model\assets


INFO:tensorflow:Assets written to: ../exported_model\bullets_model\assets


Saved model for category 'bullets' at '../exported_model\bullets_model'


INFO:tensorflow:Assets written to: ../exported_model\interesting_model\assets


INFO:tensorflow:Assets written to: ../exported_model\interesting_model\assets


...

INFO:tensorflow:Assets written to: ../exported_model\infographics_model\assets


Saved model for category 'infographics' at '../exported_model\infographics_model'


INFO:tensorflow:Assets written to: ../exported_model\graphics_model\assets


INFO:tensorflow:Assets written to: ../exported_model\graphics_model\assets


Saved model for category 'graphics' at '../exported_model\graphics_model'


In [ ]:
#LOAD the models

In [23]:
import os
import tensorflow as tf

def load_all_models(model_data_map, save_dir="saved_models"):
    for category in model_data_map.keys():
        model_path = os.path.join(save_dir, f"{category}_model")
        if os.path.exists(model_path):
            model = tf.keras.models.load_model(model_path)
            model_data_map[category]["model"] = model
            print(f"Loaded model for category '{category}' from '{model_path}'")
        else:
            print(f"Model not found for category '{category}' at '{model_path}'")

# Later, loading them back into the same map
save_dir="../exported_model"
load_all_models(model_data_map,save_dir)


Loaded model for category 'bullets' from '../exported_model\bullets_model'
Loaded model for category 'interesting' from '../exported_model\interesting_model'
Loaded model for category 'readable' from '../exported_model\readable_model'
Loaded model for category 'agenda' from '../exported_model\agenda_model'
Loaded model for category 'size_of_text' from '../exported_model\size_of_text_model'
Loaded model for category 'tables' from '../exported_model\tables_model'
Loaded model for category 'positioning' from '../exported_model\positioning_model'
Loaded model for category 'pictures' from '../exported_model\pictures_model'
Loaded model for category 'infographics' from '../exported_model\infographics_model'
Loaded model for category 'graphics' from '../exported_model\graphics_model'


## Evaluate Models

In [ ]:
def evaluation_multymodel(model_data_map, metric):
    """
    Calculate the average of a specific metric across all groups in model_data_map.

    Parameters:
    - model_data_map (dict): A dictionary where each key is a group and contains an 'evaluation' dict with metrics.
    - metric (str): The metric to average (e.g., 'accuracy', 'hamming', 'kappa_scores', 'auc_scores', 'jaccard', 'f1').

    Returns:
    - float: The average value of the specified metric across all groups.
    """
    total = 0
    count = 0

    for group, data in model_data_map.items():
        evaluation_map = data.get("evaluation", {})
        if metric in evaluation_map:
            total += evaluation_map[metric]
            count += 1

    if count == 0:
        raise ValueError(f"The metric '{metric}' was not found in any group's evaluation.")

    return total / count


def precision_recall_f1_from_cm(cm):
    TP = cm[1,1]; FP = cm[0,1]; FN = cm[1,0]
    precision = TP / (TP + FP) if (TP + FP) else 0.0
    recall    = TP / (TP + FN) if (TP + FN) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return precision, recall, f1

def aggregate_precision_recall_f1(model_data_map, groups=None):
    """
    Compute macro and micro precision/recall/F1 across groups using their confusion matrices.

    Returns:
        dict with keys:
          'macro_precision','macro_recall','macro_f1',
          'micro_precision','micro_recall','micro_f1'
    """
    if groups is None:
        groups = model_data_map.keys()

    macro_p = macro_r = macro_f1 = 0.0
    macro_count = 0

    # For micro accumulation
    TP_sum = FP_sum = FN_sum = 0

    for g in groups:
        eval_map = model_data_map[g].get("evaluation", {})
        if "confusion_matrix" not in eval_map:
            continue
        cm = eval_map["confusion_matrix"]
        p, r, f1 = precision_recall_f1_from_cm(cm)
        macro_p += p; macro_r += r; macro_f1 += f1
        macro_count += 1

        TP_sum += cm[1,1]
        FP_sum += cm[0,1]
        FN_sum += cm[1,0]

    if macro_count == 0:
        raise ValueError("No confusion_matrix found in any selected groups.")

    macro_precision = macro_p / macro_count
    macro_recall    = macro_r / macro_count
    macro_f1        = macro_f1 / macro_count

    micro_precision = TP_sum / (TP_sum + FP_sum) if (TP_sum + FP_sum) else 0.0
    micro_recall    = TP_sum / (TP_sum + FN_sum) if (TP_sum + FN_sum) else 0.0
    micro_f1        = (2 * micro_precision * micro_recall / (micro_precision + micro_recall)
                       if (micro_precision + micro_recall) else 0.0)

    return {
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'micro_precision': micro_precision,
        'micro_recall': micro_recall,
        'micro_f1': micro_f1
    }



In [ ]:
def evaluate_all_groups(model_data_map, groups=None, weighted=False):
    """
    Evaluate standard metrics across all groups.

    Parameters:
        model_data_map (dict)
        groups (Iterable[str] or None)
        weighted (bool): If True, support-weighted averages where possible.

    Returns:
        dict: aggregated metrics
    """
    scalar_metrics = {
        'hamming': 'Hamming loss',
        'kappa_scores': 'Cohen kappa',
        'auc_scores': 'ROC AUC',
        'jaccard': 'Jaccard similarity',
        'f1': 'F1 (per-group stored)',
        'accuracy': 'Accuracy'
    }

    results = {}

    for metric, label in scalar_metrics.items():
        try:
            avg_val = evaluation_multymodel_over_groups(model_data_map, metric,
                                                        groups=groups, weighted=weighted)
            results[f"{metric}_avg{'_weighted' if weighted else ''}"] = avg_val
            print(f"{label} ({'weighted' if weighted else 'macro mean'}): {avg_val:.4f}")
        except ValueError:
            print(f"[Skip] Metric '{metric}' not found in selected groups.")

    # Precision / Recall / F1 from confusion matrices
    try:
        prf = aggregate_precision_recall_f1(model_data_map, groups=groups)
        results.update(prf)
        print(f"Macro Precision: {prf['macro_precision']:.4f}")
        print(f"Macro Recall:    {prf['macro_recall']:.4f}")
        print(f"Macro F1:        {prf['macro_f1']:.4f}")
        print(f"Micro Precision: {prf['micro_precision']:.4f}")
        print(f"Micro Recall:    {prf['micro_recall']:.4f}")
        print(f"Micro F1:        {prf['micro_f1']:.4f}")
    except ValueError as e:
        print(f"[Skip] {e}")

    return results
    
# All groups, macro (simple) averages
macro_results = evaluate_all_groups(model_data_map)

# All groups, support-weighted averages for scalar metrics + macro/micro PR/F1
weighted_results = evaluate_all_groups(model_data_map, weighted=True)

## Run Training for All Labels

In [ ]:
label_columns = ['label_1', 'label_2', 'label_3']  # Replace with your actual label columns
feature_columns = [col for col in df.columns if col not in label_columns]

X, y_dict = preprocess_data(df, label_columns, feature_columns)
results_df = train_and_evaluate(X, y_dict, label_columns)
results_df


## Summary of Results

In [ ]:
results_df.sort_values(by='F1 Score', ascending=False)